In [4]:
import sys
import pandas as pd

path = "../datasets/COFACTOR-SBHUB-Oslo/Dataset_buildings_csv_files/export_8093.txt"

# 1. Find the real column-header row (the file has a metadata block above it)
with open(path, encoding="utf-8", errors="replace") as f:
    lines = f.read().splitlines()
hi = next(i for i, l in enumerate(lines) if l.startswith("TimeStamp"))
print("\n".join(lines[:hi]))          # print the metadata header

# 2. Load; timestamps are fixed UTC+1 all year (no DST), so keep them naive
df = pd.read_csv(path, sep=";", skiprows=hi)
df["ts"] = pd.to_datetime(df["TimeStamp"].str[:19])
df = df.set_index("ts").drop(columns="TimeStamp").sort_index()

Header_line;17
location;Oslo Norway
year_of_construction;1965
floor_area;9072
number_of_users;770
building_category;Sch
dhw_heat_source;DH
sh_heat_source;DH
ventilation_heat_source;DH
timestamp_format;%Y-%m-%dT%H:%M:%S%z
time_zone;Etc/Gmt-1
building_id;8093

;Heat use;Electricity import;Weather variable;Weather variable;Weather variable;Weather variable;Weather variable
;Wh;Wh;C;W/m2;m/s;deg;%
;Heat Total = HtSpace + HtDHW;Electricity imported ;Temperature Outdoor;Global Solar Horizontal Radiation;Wind_speed;Wind_dir;Relative Humidity


In [5]:

# 3. Basic shape and gaps
print(f"\nrows={len(df)}  {df.index.min()} -> {df.index.max()}")
full = pd.date_range(df.index.min(), df.index.max(), freq="h")
print("missing hours:", len(full.difference(df.index)),
      "| duplicates:", df.index.duplicated().sum())
print(df.describe().T[["count", "min", "mean", "max"]])



rows=8809  2019-03-05 00:00:00 -> 2020-03-06 00:00:00
missing hours: 0 | duplicates: 0
          count         min           mean            max
HtTot    8809.0    0.000000  100979.675332  425600.000000
ElImp    8809.0    0.000000  220486.418095  422000.000000
Tout     8809.0 -273.150000       7.882071      32.776300
SolGlob  8809.0   -0.067083     105.056946     817.616389
WindSpd  8809.0    0.137192       2.843351      10.164894
WindDir  8809.0    0.133946     155.308542     359.901120
RH       8809.0    0.279040       0.774073       0.997093


In [6]:

# 4. Known quality issues
print("\nTout sentinel (<-100):", df.index[df["Tout"] < -100].tolist())
print("negative SolGlob hours:", (df["SolGlob"] < 0).sum())
print("HtTot == 0 hours:", (df["HtTot"] == 0).sum())
flat = df["HtTot"].eq(df["HtTot"].shift()) & df["HtTot"].gt(0)
print("HtTot repeated-value hours:", flat.sum())



Tout sentinel (<-100): [Timestamp('2020-02-20 09:00:00')]
negative SolGlob hours: 386
HtTot == 0 hours: 117
HtTot repeated-value hours: 1485


In [ ]:

# 5. Clean copy (Wh/h == average W)
c = df.copy()
c.loc[c["Tout"] < -100, "Tout"] = pd.NA
c["Tout"] = c["Tout"].astype(float).interpolate(limit=3)
c["SolGlob"] = c["SolGlob"].clip(lower=0)
c.loc[c["HtTot"].eq(5000), "HtTot"] = pd.NA   # flat stretch around 2019-11-23/24

# 6. Quick sanity: monthly heat, and the Jan-Feb 2020 candidate window
print("\nHtTot MWh per month:")
print((c["HtTot"].resample("MS").sum() / 1e6).round(1))
w = c.loc["2020-01-06":"2020-02-04"]
print("\nwindow rows:", len(w), "| NaNs:\n", w.isna().sum())
print("avg weekday kW by hour:\n",
      (w[w.index.dayofweek < 5].groupby(w[w.index.dayofweek < 5].index.hour)["HtTot"].mean() / 1e3).round(0).to_string())


HtTot MWh per month:
ts
2019-03-01    146.6
2019-04-01    105.0
2019-05-01     82.6
2019-06-01     34.6
2019-07-01     20.7
2019-08-01     26.3
2019-09-01     33.3
2019-10-01     84.6
2019-11-01    115.6
2019-12-01     80.0
2020-01-01     75.6
2020-02-01     67.9
2020-03-01     16.7
Freq: MS, Name: HtTot, dtype: float64

window rows: 720 | NaNs:
 HtTot      0
ElImp      0
Tout       0
SolGlob    0
WindSpd    0
WindDir    0
RH         0
dtype: int64
avg weekday kW by hour:
 ts
0      53.0
1      55.0
2      57.0
3      57.0
4      60.0
5      69.0
6     131.0
7     206.0
8     205.0
9     187.0
10    184.0
11    174.0
12    159.0
13    152.0
14    164.0
15    159.0
16    143.0
17    143.0
18    139.0
19    132.0
20    104.0
21     70.0
22     50.0
23     47.0
